# Technical Analysis: Indicators & Chart Patterns

**Category:** 11-Financial  
**Description:** Moving averages, RSI, MACD, Bollinger Bands  
**Libraries:** Pandas, NumPy, Matplotlib

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy matplotlib mplfinance

CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
MODEL = CLAUDE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

# Part 1: Generate OHLC Data

---

In [ ]:
def generate_ohlc(n_days=500, start_price=100, volatility=0.02, trend=0.0001):
    """
    Generate synthetic OHLC (Open, High, Low, Close) data.
    """
    dates = pd.date_range('2023-01-01', periods=n_days, freq='B')
    
    # Generate log returns with trend
    returns = np.random.randn(n_days) * volatility + trend
    
    # Generate close prices
    close = start_price * np.exp(np.cumsum(returns))
    
    # Generate OHLC from close
    intraday_vol = volatility * 0.5
    
    high = close * (1 + np.abs(np.random.randn(n_days) * intraday_vol))
    low = close * (1 - np.abs(np.random.randn(n_days) * intraday_vol))
    
    # Open is previous close with gap
    open_prices = np.roll(close, 1) * (1 + np.random.randn(n_days) * 0.002)
    open_prices[0] = start_price
    
    # Ensure OHLC consistency
    high = np.maximum(high, np.maximum(open_prices, close))
    low = np.minimum(low, np.minimum(open_prices, close))
    
    # Volume
    volume = np.random.lognormal(15, 0.5, n_days).astype(int)
    
    df = pd.DataFrame({
        'Open': open_prices,
        'High': high,
        'Low': low,
        'Close': close,
        'Volume': volume
    }, index=dates)
    
    return df

# Generate data
df = generate_ohlc(n_days=500, start_price=100, volatility=0.015, trend=0.0003)

print("OHLC Data Sample:")
df.tail(10)

---

# Part 2: Moving Averages

---

In [ ]:
# Simple Moving Averages (SMA)
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# Exponential Moving Average (EMA)
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# Price with MAs
axes[0].plot(df.index, df['Close'], 'gray', linewidth=1, alpha=0.7, label='Close')
axes[0].plot(df.index, df['SMA_20'], 'b-', linewidth=1.5, label='SMA 20')
axes[0].plot(df.index, df['SMA_50'], 'orange', linewidth=1.5, label='SMA 50')
axes[0].plot(df.index, df['SMA_200'], 'r-', linewidth=2, label='SMA 200')
axes[0].set_title('Price with Moving Averages')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df.index, df['Volume'], color='steelblue', alpha=0.7, width=0.8)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Moving Average Crossover Signals
df['Signal'] = 0
df.loc[df['SMA_20'] > df['SMA_50'], 'Signal'] = 1  # Bullish
df.loc[df['SMA_20'] < df['SMA_50'], 'Signal'] = -1  # Bearish

# Detect crossovers
df['Crossover'] = df['Signal'].diff()

# Golden Cross (bullish) and Death Cross (bearish)
golden_crosses = df[df['Crossover'] == 2].index
death_crosses = df[df['Crossover'] == -2].index

# Plot with signals
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df.index, df['Close'], 'gray', linewidth=1, alpha=0.5)
ax.plot(df.index, df['SMA_20'], 'b-', linewidth=1.5, label='SMA 20')
ax.plot(df.index, df['SMA_50'], 'orange', linewidth=1.5, label='SMA 50')

# Mark crossovers
for gc in golden_crosses:
    ax.axvline(gc, color='green', linestyle='--', alpha=0.5)
    ax.scatter(gc, df.loc[gc, 'Close'], color='green', s=100, marker='^', zorder=5)

for dc in death_crosses:
    ax.axvline(dc, color='red', linestyle='--', alpha=0.5)
    ax.scatter(dc, df.loc[dc, 'Close'], color='red', s=100, marker='v', zorder=5)

ax.scatter([], [], color='green', marker='^', s=100, label='Golden Cross (Buy)')
ax.scatter([], [], color='red', marker='v', s=100, label='Death Cross (Sell)')

ax.set_title('Moving Average Crossover Strategy')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Golden Crosses: {len(golden_crosses)}")
print(f"Death Crosses: {len(death_crosses)}")

---

# Part 3: RSI (Relative Strength Index)

---

In [ ]:
def calculate_rsi(prices, period=14):
    """
    Calculate Relative Strength Index.
    """
    delta = prices.diff()
    
    gains = delta.copy()
    losses = delta.copy()
    
    gains[gains < 0] = 0
    losses[losses > 0] = 0
    losses = abs(losses)
    
    avg_gain = gains.rolling(window=period).mean()
    avg_loss = losses.rolling(window=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

df['RSI'] = calculate_rsi(df['Close'], period=14)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# Price
axes[0].plot(df.index, df['Close'], 'b-', linewidth=1)
axes[0].set_title('Price and RSI')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df.index, df['RSI'], 'purple', linewidth=1)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].axhline(50, color='gray', linestyle=':', alpha=0.5)
axes[1].fill_between(df.index, df['RSI'], 70, where=(df['RSI'] >= 70), alpha=0.3, color='red')
axes[1].fill_between(df.index, df['RSI'], 30, where=(df['RSI'] <= 30), alpha=0.3, color='green')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 4: MACD (Moving Average Convergence Divergence)

---

In [ ]:
def calculate_macd(prices, fast=12, slow=26, signal=9):
    """
    Calculate MACD indicator.
    """
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    
    return macd_line, signal_line, histogram

df['MACD'], df['Signal_Line'], df['MACD_Hist'] = calculate_macd(df['Close'])

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# Price
axes[0].plot(df.index, df['Close'], 'b-', linewidth=1)
axes[0].set_title('Price and MACD')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# MACD
axes[1].plot(df.index, df['MACD'], 'b-', linewidth=1.5, label='MACD')
axes[1].plot(df.index, df['Signal_Line'], 'r-', linewidth=1.5, label='Signal')

# Histogram
colors = ['green' if h >= 0 else 'red' for h in df['MACD_Hist']]
axes[1].bar(df.index, df['MACD_Hist'], color=colors, alpha=0.5, width=0.8)

axes[1].axhline(0, color='gray', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('MACD')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 5: Bollinger Bands

---

In [ ]:
def calculate_bollinger_bands(prices, window=20, num_std=2):
    """
    Calculate Bollinger Bands.
    """
    middle = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
    
    upper = middle + (std * num_std)
    lower = middle - (std * num_std)
    
    # Bandwidth and %B
    bandwidth = (upper - lower) / middle * 100
    percent_b = (prices - lower) / (upper - lower)
    
    return middle, upper, lower, bandwidth, percent_b

df['BB_Middle'], df['BB_Upper'], df['BB_Lower'], df['BB_Width'], df['BB_PercentB'] = \
    calculate_bollinger_bands(df['Close'])

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1, 1]})

# Price with Bollinger Bands
axes[0].plot(df.index, df['Close'], 'b-', linewidth=1, label='Close')
axes[0].plot(df.index, df['BB_Middle'], 'orange', linewidth=1, label='Middle Band')
axes[0].plot(df.index, df['BB_Upper'], 'gray', linewidth=1, linestyle='--')
axes[0].plot(df.index, df['BB_Lower'], 'gray', linewidth=1, linestyle='--')
axes[0].fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.2, color='gray')
axes[0].set_title('Bollinger Bands')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Bandwidth
axes[1].plot(df.index, df['BB_Width'], 'purple', linewidth=1)
axes[1].set_ylabel('Bandwidth (%)')
axes[1].grid(True, alpha=0.3)

# %B
axes[2].plot(df.index, df['BB_PercentB'], 'teal', linewidth=1)
axes[2].axhline(1, color='red', linestyle='--', alpha=0.5)
axes[2].axhline(0, color='green', linestyle='--', alpha=0.5)
axes[2].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[2].set_ylabel('%B')
axes[2].set_ylim(-0.5, 1.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
%%ai $MODEL
Explain these technical indicators to a new trader:

1. Moving Average Crossovers: What do golden/death crosses signal?
2. RSI: How should traders interpret overbought/oversold conditions?
3. MACD: What does the histogram tell us about momentum?
4. Bollinger Bands: When do they squeeze and what does it mean?

Include common mistakes traders make with each indicator.

---

# Part 6: Combined Dashboard

---

In [ ]:
# Create a comprehensive technical analysis dashboard
recent = df.tail(100).copy()  # Last 100 days

fig, axes = plt.subplots(4, 1, figsize=(14, 14), 
                          gridspec_kw={'height_ratios': [3, 1, 1, 1]})

# Panel 1: Price with Bollinger Bands & Moving Averages
ax1 = axes[0]
ax1.plot(recent.index, recent['Close'], 'k-', linewidth=1.5, label='Close')
ax1.plot(recent.index, recent['SMA_20'], 'b-', linewidth=1, alpha=0.7, label='SMA 20')
ax1.plot(recent.index, recent['SMA_50'], 'orange', linewidth=1, alpha=0.7, label='SMA 50')
ax1.fill_between(recent.index, recent['BB_Upper'], recent['BB_Lower'], 
                  alpha=0.2, color='gray', label='Bollinger Bands')
ax1.set_title('Technical Analysis Dashboard', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Panel 2: Volume
ax2 = axes[1]
colors = ['green' if recent['Close'].iloc[i] >= recent['Open'].iloc[i] else 'red' 
          for i in range(len(recent))]
ax2.bar(recent.index, recent['Volume'], color=colors, alpha=0.7, width=0.8)
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)

# Panel 3: RSI
ax3 = axes[2]
ax3.plot(recent.index, recent['RSI'], 'purple', linewidth=1)
ax3.axhline(70, color='red', linestyle='--', alpha=0.5)
ax3.axhline(30, color='green', linestyle='--', alpha=0.5)
ax3.fill_between(recent.index, recent['RSI'], 70, 
                  where=(recent['RSI'] >= 70), alpha=0.3, color='red')
ax3.fill_between(recent.index, recent['RSI'], 30, 
                  where=(recent['RSI'] <= 30), alpha=0.3, color='green')
ax3.set_ylabel('RSI')
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.3)

# Panel 4: MACD
ax4 = axes[3]
ax4.plot(recent.index, recent['MACD'], 'b-', linewidth=1, label='MACD')
ax4.plot(recent.index, recent['Signal_Line'], 'r-', linewidth=1, label='Signal')
colors = ['green' if h >= 0 else 'red' for h in recent['MACD_Hist']]
ax4.bar(recent.index, recent['MACD_Hist'], color=colors, alpha=0.5, width=0.8)
ax4.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax4.set_ylabel('MACD')
ax4.set_xlabel('Date')
ax4.legend(loc='upper left', fontsize=8)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Current Signal Summary
latest = df.iloc[-1]

print("="*60)
print("TECHNICAL ANALYSIS SUMMARY")
print("="*60)
print(f"\nDate: {df.index[-1].strftime('%Y-%m-%d')}")
print(f"Close: ${latest['Close']:.2f}")
print()

# Moving Averages
print("Moving Averages:")
print(f"  SMA 20:  ${latest['SMA_20']:.2f} {'(Above)' if latest['Close'] > latest['SMA_20'] else '(Below)'}")
print(f"  SMA 50:  ${latest['SMA_50']:.2f} {'(Above)' if latest['Close'] > latest['SMA_50'] else '(Below)'}")
print(f"  SMA 200: ${latest['SMA_200']:.2f} {'(Above)' if latest['Close'] > latest['SMA_200'] else '(Below)'}")

# RSI
rsi_signal = 'Overbought' if latest['RSI'] > 70 else ('Oversold' if latest['RSI'] < 30 else 'Neutral')
print(f"\nRSI: {latest['RSI']:.1f} ({rsi_signal})")

# MACD
macd_signal = 'Bullish' if latest['MACD'] > latest['Signal_Line'] else 'Bearish'
print(f"MACD: {macd_signal} (MACD: {latest['MACD']:.3f}, Signal: {latest['Signal_Line']:.3f})")

# Bollinger Bands
bb_position = 'Upper Band' if latest['BB_PercentB'] > 1 else ('Lower Band' if latest['BB_PercentB'] < 0 else 'Within Bands')
print(f"Bollinger %B: {latest['BB_PercentB']:.2f} ({bb_position})")

---

## Summary

This notebook covered:
- Moving Averages (SMA, EMA) and crossover signals
- Relative Strength Index (RSI) for momentum
- MACD for trend following
- Bollinger Bands for volatility analysis
- Combined technical analysis dashboard

**Note:** Technical analysis should be used alongside fundamental analysis and proper risk management.

---